<a href="https://colab.research.google.com/github/ShiX27/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


**Code Explanation:** First, upload `product_catalog.csv` to Google Colab and import `pandas` as `pd`. Second, set the file path to `'/content/product_catalog.csv'`. Then, use a `try` statement to read the CSV file from the specified path and store it in the `cat` DataFrame. Third, use an `except` `FileNotFoundError` statement to load the sample dataset if the file cannot be found. Finally, display the first five rows of the `cat` DataFrame. In this case, the file was found, so the output displays the actual dataset.

### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
filtered = cat[cat['product_name'].str.lower().str.contains('pop')]
print(f'Count: {len(filtered)}')
filtered

Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


There are 4 rows that look like Pop-Tart products based on product names containing "pop".

**Code Explanation:** First, filter the `product_name` column by converting the product names to lowercase and selecting the rows containing '`pop`'. Store the results in `filtered`. Then, count the number of rows in `filtered` and print the count. Finally, display the `filtered` DataFrame.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = filtered[filtered['product_name'].str.lower().str.contains('strawberry')]
non_strawberry = filtered[~filtered['product_name'].str.lower().str.contains('strawberry')]

print('Strawberry Pop-Tarts:')
print(strawberry[['sku', 'product_name']])

print('Non-Strawberry Pop-Tarts:')
print(non_strawberry[['sku', 'product_name']])

Strawberry Pop-Tarts:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry
Non-Strawberry Pop-Tarts:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** There are no Pop-Tart rows representing a different product, since all four rows refer to strawberry Pop-Tarts.

The four SKUs and product names that all refer to strawberry Pop-Tarts are PT-12 (pop-tarts strawberry), SKU#459812 (Strawberry Pop-Tart), POPTART-STRAW (Pop-Tarts Strawberry), and PT_STRAWBERRY (pop-tarts strawberry). There are no non-strawberry Pop-Tart products. Any non-strawberry flavors should remain separate because they represent different products.

**Code Explanation:** First, use the `filtered` DataFrame from Q1 and select the rows containing '`strawberry`' in their product names after converting them to lowercase. Store the results in the `strawberry` DataFrame. Second, use `~` to select the rows with product names that do not contain '`strawberry`' and store the results in the `non_straberry` DataFrame. Finally, display the `sku` and `product_name` columns for both groups.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW'
}
def consolidate(df):
  out = df.copy()
  out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
  canonical = out.pop('canonical_sku')
  out.insert(1, 'canonical_sku', canonical)
  return out

original_rows = len(cat)
cat = consolidate(cat)
consolidated_rows = len(cat)
cat.head()

,sku,canonical_sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,POPTART-STRAW,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,POPTART-STRAW,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,POPTART-STRAW,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


**Code Explanation:** First, create a dictionary called `CANON` to map all strawberry Pop-Tart SKUs to one canonical SKU, `POPTART-STRAW`. Second, create a function called `consolidate()`. Inside the function, create a copy of the DataFrame and call it `out`, map the SKUs using the `CANON` dictionary, and keep the original SKUs if they are not included in the dictionary. Store the results in a new column called `canonical_sku`. Then, temporarily remove the `canonical_sku` column from out and store its values in `canonical`. Insert the `canonical_sku` column back into `out` at position 1, immediately after the `sku` column. Return `out`. Thrid, store the number of rows in the original `cat` DataFrame as `orignal_rows`, and apply the `consolidate()` function to `cat`, and store the result back in `cat`. Then, store the number of rows after consolidation as `consolidated_rows`. Finally, display the first five rows of the `cat` DataFrame.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# Before normalization
print('Before normalization:')
print(cat['category'].value_counts(dropna=False))

# Normalize category
cat['category'] = cat['category'].str.strip().str.title()

# After normalization
print('\nAfter normalization:')
print(cat['category'].value_counts(dropna=False))

Before normalization:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After normalization:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health      

After normalization, categories with different capitalization were combined into the same labels. However, some categories still have different names but represent the same category, such as "Emergency" and "Emergency Supplies", "Grocery" and "Groceries", "Breakfast" and "Breakfast Foods", "Baby" and "Baby Products", and "Alcohol" and "Alcoholic Beverages."

**Code Explanation:** First, print the counts for each distinct category before normalization, including missing values (NaN) in the counts. Second, normalize the categories by removing leading and trailing spaces and capitalizing the first letter of each word. Finally, print the counts for each distinct category after normalization, including missing values.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
cat['normalized_name'] = (cat['product_name'].str.lower()
                          .str.replace(r'[^\w\s]|_', '', regex=True)
                          .str.split().apply(lambda w: ' '.join(sorted(w))))
normalized_name = (cat.groupby('normalized_name')['sku']
                   .agg('unique').loc[lambda s: s.str.len() > 1])   # .unique() returns only the distinct SKU values in each group
normalized_name

,sku
normalized_name,
24pack beer light,"[BEER-24, LTBEER24]"
4pack aa batteries,"[BAT-AA4, BATT_AA_4PK]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


The table does not include the products, including bottled watter (24-pack), baby diapers (Size 3), baby formula or infant formula, first aid kits, paper towels (12-roll), portable phone charger, and Roku streaming sticks. This is because their product names differ in wording, abrreviations, or punctuation, such as hyphens. Even after normalization, these products still have different normalized names, so they are treated as separate groups with only one SKU each and are not included in the table.

**Code Explanation:** First, create a new column called `normalized_name` by converting the `product_name` column to be lowercase, romoving punctuation (including hyphens and underscores) while keeping letters, numbers, and spaces,  splitting the names into words, sorting the words alphabetically, and joining them back into strings. Second, group the products by their normalized names and `use .agg('unique')` to collect the distinct SKUs in each group. Third, use `.loc[lambda s: s.str.len() > 1]` to keep only groups with more than one distinct SKU. Store the results in the `normalized_name` variable. Finally, display the results.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all()
assert original_rows == consolidated_rows
print('checks passed.')

checks passed.


**Code Explanation:** Use two `assert` statements to check that there are no missing values in the `canonical_sku` column and that the number of rows does not change before and after consolidation using the variables created in Q3. If both assertions pass, print `'checks passed.'`

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families that appear fragmented are Monopoly board games and canned soup. Monopoly board games have three different SKUs (BG-MONO, MONOPOLY-01, and GAME-MONO), and canned soup has two different SKUs (SOUP-CAN and CNSOUP-01). I would group the SKUs for each product under one canonical SKU, such as GAME-MONO and SOUP-CAN, respectively, because they appear to represent the same products but have different SKUs and product names.

b) Consolidation becomes a business decision when products have similar names but may not represent the same product. For example, I am not sure whether baby formula and infant formula should be grouped together because they may differ in brand, age range, or package size. I would ask the inventory manager to confirm whether they should be treated as the same product or kept separate before grouping them.

Project 1 Group Members: Xiangyu Shi and Simi Chakravarty.